In [ ]:
df_raw = spark.read.format("json").load(
    "/Volumes/osrs_analytics_prod/raw/landing/ge_price_24h/*"
)

In [0]:
from pyspark.sql.functions import explode, col, from_json, to_json

df = (
    df_raw
    .withColumn(
        "data_map",
        from_json(
            to_json(col("data")),
            "map<string,map<string,string>>"
        )
    )

    .select(
        explode(col("data_map")).alias("item_id", "price_metrics"),
        col("timestamp")
    )
    
    .select(
        col("item_id").cast("bigint").alias("item_id"),
        col("price_metrics")["avgHighPrice"].cast("bigint").alias("buy_price"),
        col("price_metrics")["avgLowPrice"].cast("bigint").alias("sell_price"),
        col("price_metrics")["highPriceVolume"].cast("bigint").alias("buy_volume"),
        col("price_metrics")["lowPriceVolume"].cast("bigint").alias("sell_volume"),
        col("timestamp")
    )
)

In [0]:
from pyspark.sql.functions import (
    col, lit, log, greatest,
    from_unixtime, to_timestamp, date_format, to_date
)

df_base = df.withColumns({
    "date": to_date(from_unixtime(col("timestamp"))), 
    "timestamp_dt": to_timestamp(from_unixtime(col("timestamp"))),

    "datetime_str": date_format(
        to_timestamp(from_unixtime(col("timestamp"))),
        "yyyy-MM-dd HH:mm:ss"
    ),

    "mid_price": ((col("buy_price") + col("sell_price")) / 2).cast("double"),
    
    "spread_abs": (col("buy_price") - col("sell_price")).cast("double"),
    "spread_pct": (
        (col("buy_price") - col("sell_price")) /
        col("sell_price")
    ).cast("double"),

    "total_volume": (
        col("buy_volume") + col("sell_volume")
    ).cast("double"),

    "buy_pressure": (
        col("buy_volume") /
        greatest(
            col("buy_volume") + col("sell_volume"),
            lit(1)
        )
    ).cast("double"),

})

In [0]:
df_base = (
    df_base
    .repartition("date")  
    .sortWithinPartitions("item_id", "timestamp")
)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    avg, stddev, max as spark_max, min as spark_min,
    count, col, lag, lead, lit, greatest
)

def days_sec(n):
    return n * 86400


item_w = Window.partitionBy("item_id").orderBy("timestamp")


windows = {
    "1":  days_sec(1),
    "7":  days_sec(7),
    "14": days_sec(14),
    "30": days_sec(30),
    "90": days_sec(90),
}


df_features = df_base


# --- TIME-BASED FEATURES ---
for label, seconds in windows.items():
    n = int(label)
    w = item_w.rangeBetween(-seconds, 0)

    df_features = df_features.withColumns({
        f"buy_price_lag_{label}d": lag("buy_price", n).over(item_w),
        f"buy_price_ma_{label}d": avg("buy_price").over(w),
        f"buy_price_std_{label}d": stddev("buy_price").over(w),
        f"buy_price_max_{label}d": spark_max("buy_price").over(w),
        f"buy_price_min_{label}d": spark_min("buy_price").over(w),

        f"sell_price_lag_{label}d": lag("sell_price", n).over(item_w),
        f"sell_price_ma_{label}d": avg("sell_price").over(w),
        f"sell_price_std_{label}d": stddev("sell_price").over(w),
        f"sell_price_max_{label}d": spark_max("sell_price").over(w),
        f"sell_price_min_{label}d": spark_min("sell_price").over(w),

        f"mid_price_lag_{label}d": lag("mid_price", n).over(item_w),
        f"mid_price_ma_{label}d": avg("mid_price").over(w),
        f"mid_price_std_{label}d": stddev("mid_price").over(w),
        f"mid_price_max_{label}d": spark_max("mid_price").over(w),
        f"mid_price_min_{label}d": spark_min("mid_price").over(w),

        f"buy_volume_lag_{label}d": lag("buy_volume", n).over(item_w),
        f"buy_volume_ma_{label}d": avg("buy_volume").over(w),
        f"buy_volume_std_{label}d": stddev("buy_volume").over(w),
        f"buy_volume_max_{label}d": spark_max("buy_volume").over(w),
        f"buy_volume_min_{label}d": spark_min("buy_volume").over(w),

        f"sell_volume_lag_{label}d": lag("sell_volume", n).over(item_w),
        f"sell_volume_ma_{label}d": avg("sell_volume").over(w),
        f"sell_volume_std_{label}d": stddev("sell_volume").over(w),
        f"sell_volume_max_{label}d": spark_max("sell_volume").over(w),
        f"sell_volume_min_{label}d": spark_min("sell_volume").over(w),

        f"buy_pressure_lag_{label}d": lag("buy_pressure", n).over(item_w),
        f"buy_pressure_ma_{label}d": avg("buy_pressure").over(w),
        f"snapshot_count_{label}d": count("timestamp").over(w),
    })
# --- DERIVED FEATURES ---
df_features = df_features.withColumns({

    "notional_volume": col("total_volume") * col("mid_price"),
})

In [0]:
# -----------------------------------------------------------------------------
# Forward target horizons
# -----------------------------------------------------------------------------
forward_windows = {
    "1":  days_sec(1),
    "7":  days_sec(7),
    "14": days_sec(14),
    "30": days_sec(30),
    "90": days_sec(90),
}


# -----------------------------------------------------------------------------
# Attach future prices
# -----------------------------------------------------------------------------
for label, seconds in windows.items():
    n = int(label)
    w = item_w.rangeBetween(-seconds, 0)

    df_targets = df_features.withColumns({
        f"buy_price_lead_{label}d": lead("buy_price", n).over(item_w)
    })
 

In [0]:
path = "/Volumes/osrs_analytics_dev/dev/ml/price_forecast/build"

(
    df_targets
    .write
    .option("compression", "zstd")
    .partitionBy("date")
    .mode("overwrite")
    .parquet(path)
)